# 05 — Stage 1b: Passive Benchmark and Cash-Drag Correction

R̃<sub>i,t</sub> = R<sub>i,t</sub> + Cash<sub>i,t</sub> · (R<sub>passive,t</sub> − r<sub>f,t</sub>),
with Cash<sub>i,t</sub> = `per_cash_clean` / 100 and R<sub>passive,t</sub> the equal-weighted return of CRSP pure index funds
(`index_fund_flag == "D"`) in the fund's Lipper objective category in month *t*.

| | |
|---|---|
| **Inputs** | CRSP `fund_summary2`, `monthly_tna_ret_nav` (via the NB01 Parquet cache); `panel_with_factors.parquet` (NB03) |
| **Outputs** | `rpassive_D.parquet` (category × month benchmark), `stage1b_corrected_returns.parquet` → read by NB06, NB07 |
| **Thesis** | Sections 4.1 (passive alpha), 4.3, 5.3 (Table 4), 5.4.1 (EW vs VW), 5.4.2 (alternative cash instrument) |
| **Expected** | benchmark coverage 98.3% (D) / 99.2% (D+B); Table 4 on 326,365 correctable fund-months |

**Net returns.** Both R<sub>i,t</sub> and R<sub>passive,t</sub> are CRSP net returns (after expense ratios and 12b-1 fees),
so the benchmark carries the index funds' fees; this is why its factor-model alpha (Section 3) is negative.

**To replicate:** set `DATA_ROOT` in `thesis_config.py`, run NB01–NB03, then *Restart Kernel → Run All*.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from thesis_config import DATA_DIR, FACTORS, HAC_LAGS, assert_matches, check_paths, read_sas_cached, save_parquet

check_paths(DATA_DIR / "panel_with_factors.parquet")
print(f"pandas {pd.__version__} | numpy {np.__version__} | statsmodels {sm.__version__}")

## 1. Passive benchmark R<sub>passive</sub>

Passive funds: `index_fund_flag` ∈ {D, B}, `lipper_asset_cd == "EQ"`, and not a strategy whose market beta is
structurally ≠ 1 (short-bias, market-neutral, leveraged, commodity, convertible, …). Each passive fund gets its
most recent flag and its most recent non-empty Lipper code. The primary benchmark uses pure index funds (D) only;
D+B is computed for the coverage comparison in Section 4.3.

In [ ]:
df2 = read_sas_cached("fund_summary2.sas7bdat",
                      columns=["crsp_fundno", "caldt", "index_fund_flag", "lipper_asset_cd", "lipper_obj_cd"])
df2["crsp_fundno"] = df2["crsp_fundno"].dropna().astype(int).astype(str).str.zfill(6)
df2["caldt"] = pd.to_datetime(df2["caldt"])

monthly = read_sas_cached("monthly_tna_ret_nav.sas7bdat")
monthly["crsp_fundno"] = monthly["crsp_fundno"].dropna().astype(int).astype(str).str.zfill(6)
monthly["caldt"] = pd.to_datetime(monthly["caldt"])

NON_EQUITY_BETA = ["DSB", "EMN", "CMD", "CAG", "CV", "MFF", "ABR", "I", "AMS", "RR", "OS"]
_pass = df2[df2["index_fund_flag"].isin(["D", "B"]) &
            (df2["lipper_asset_cd"] == "EQ") &
            ~df2["lipper_obj_cd"].isin(NON_EQUITY_BETA)].copy()

passive_flag = (_pass[["crsp_fundno", "caldt", "index_fund_flag"]]
                .sort_values("caldt").groupby("crsp_fundno", as_index=False).last()
                [["crsp_fundno", "index_fund_flag"]])
passive_cat = (_pass[_pass["lipper_obj_cd"].notna() & (_pass["lipper_obj_cd"].astype(str).str.strip() != "")]
               [["crsp_fundno", "caldt", "lipper_obj_cd"]]
               .sort_values("caldt").groupby("crsp_fundno", as_index=False).last()
               [["crsp_fundno", "lipper_obj_cd"]])
passive_meta = passive_flag.merge(passive_cat, on="crsp_fundno", how="left")
assert passive_meta["crsp_fundno"].is_unique

passive_all = monthly.merge(passive_meta, on="crsp_fundno", how="inner", validate="many_to_one")
passive_D   = passive_all[passive_all["index_fund_flag"] == "D"].copy()
passive_DB  = passive_all.copy()

# Equal-weighted benchmark return per (month, Lipper category)
rpassive_D  = (passive_D.groupby(["caldt", "lipper_obj_cd"])["mret"].mean()
               .reset_index().rename(columns={"mret": "r_passive"}))
rpassive_DB = (passive_DB.groupby(["caldt", "lipper_obj_cd"])["mret"].mean()
               .reset_index().rename(columns={"mret": "r_passive"}))
print(f"Passive D:   {passive_D['crsp_fundno'].nunique():,} funds, {len(rpassive_D):,} category-months")
print(f"Passive D+B: {passive_DB['crsp_fundno'].nunique():,} funds, {len(rpassive_DB):,} category-months")

**Match to the active panel** on (year-month, Lipper objective code).

In [ ]:
active_panel = pd.read_parquet(DATA_DIR / "panel_with_factors.parquet")
active_panel["year_month"] = pd.to_datetime(active_panel["caldt"]).dt.to_period("M")
rpassive_D["year_month"]   = pd.to_datetime(rpassive_D["caldt"]).dt.to_period("M")
rpassive_DB["year_month"]  = pd.to_datetime(rpassive_DB["caldt"]).dt.to_period("M")

merged_D  = active_panel.merge(rpassive_D[["year_month", "lipper_obj_cd", "r_passive"]],
                               on=["year_month", "lipper_obj_cd"], how="left")
merged_DB = active_panel.merge(rpassive_DB[["year_month", "lipper_obj_cd", "r_passive"]],
                               on=["year_month", "lipper_obj_cd"], how="left")
cov_D, cov_DB = merged_D["r_passive"].notna().mean(), merged_DB["r_passive"].notna().mean()
print(f"Active fund-months matched to a passive benchmark: D {cov_D:.1%} | D+B {cov_DB:.1%}")
print(f"Active categories without any D benchmark: "
      f"{set(active_panel['lipper_obj_cd'].dropna()) - set(passive_D['lipper_obj_cd'].dropna()) or 'none'}")

# Spot check: a benchmark value equals the plain mean of its passive funds' returns
cat, ym = "G", merged_D.loc[merged_D["lipper_obj_cd"] == "G", "year_month"].value_counts().index[0]
manual = passive_D.loc[(passive_D["lipper_obj_cd"] == cat) & (passive_D["caldt"].dt.to_period("M") == ym), "mret"].mean()
attached = merged_D.loc[(merged_D["lipper_obj_cd"] == cat) & (merged_D["year_month"] == ym), "r_passive"]
assert np.allclose(attached, manual, equal_nan=True)

assert_matches("Section 4.3: benchmark coverage D, D+B (%)", [cov_D * 100, cov_DB * 100], [98.3, 99.2], 1)

save_parquet(rpassive_D, DATA_DIR / "rpassive_D.parquet")

## 2. Cash-corrected returns

Fund-months without a cash weight or without a benchmark keep `r_corrected = NaN`; they are excluded pairwise in
Stage 2 (NB06). `per_cash_clean` is missing almost entirely before 2003 (see NB01 appendix).

In [ ]:
merged_D["cash_weight"]        = merged_D["per_cash_clean"] / 100
merged_D["r_corrected"]        = merged_D["mret"] + merged_D["cash_weight"] * (merged_D["r_passive"] - merged_D["rf"])
merged_D["excess_r_corrected"] = merged_D["r_corrected"] - merged_D["rf"]

miss = merged_D["r_corrected"].isna()
print(f"r_corrected missing: {miss.sum():,} ({miss.mean():.1%}) — "
      f"no cash weight only {(miss & merged_D['r_passive'].notna()).sum():,}, "
      f"no benchmark only {(miss & merged_D['per_cash_clean'].notna()).sum():,}, "
      f"both {(merged_D['r_passive'].isna() & merged_D['per_cash_clean'].isna()).sum():,}")

save_parquet(merged_D, DATA_DIR / "stage1b_corrected_returns.parquet", index=False)

**Table 4 — cash-correction magnitude** (basis points per month, correctable fund-months).

In [ ]:
common = merged_D["r_corrected"].notna() & merged_D["mret"].notna()
bps = pd.DataFrame({
    "Raw return":       merged_D.loc[common, "mret"] * 1e4,
    "Corrected return": merged_D.loc[common, "r_corrected"] * 1e4,
    "Correction":      (merged_D.loc[common, "r_corrected"] - merged_D.loc[common, "mret"]) * 1e4,
})
table4 = bps.agg(["mean", "median", "std", "min", "max", "count"]).T
print(table4.round(1).to_string())

THESIS_TABLE4 = {   #                mean  median    std     min     max       N
    "Raw return":       ( 84.9,  129.4, 503.5, -8876, 9091, 326_365),
    "Corrected return": ( 86.5,  132.2, 514.7, -9009, 9075, 326_365),
    "Correction":       (  1.6,    0.3,  26.3, -1602, 1280, 326_365),
}
assert_matches("Section 5.3: correction mean, median (bps/month)",
               [bps["Correction"].mean(), bps["Correction"].median()], [1.63, 0.28], 2)
for row, exp in THESIS_TABLE4.items():
    assert_matches(f"Table 4 {row}: mean, median, std", table4.loc[row, ["mean", "median", "std"]], exp[:3], 1)
    assert_matches(f"Table 4 {row}: min, max, N", table4.loc[row, ["min", "max", "count"]].round(), exp[3:], 0)

## 3. Benchmark diagnostics

**Passive benchmark alpha (context for Section 4.1).** Each category's benchmark excess return is regressed on the
seven factors (HAC 12) over the sample window, for the categories present in the active panel; categories are
aggregated by their share of active fund-months. The "trusted core" keeps diversified categories with R² ≥ 0.90
and a median of ≥ 15 passive funds per month.

In [ ]:
MIN_MONTHS = 60
panel_cats     = set(active_panel["lipper_obj_cd"].dropna().unique())
ym_min, ym_max = active_panel["year_month"].min(), active_panel["year_month"].max()
fac = active_panel[["year_month"] + FACTORS + ["rf"]].drop_duplicates("year_month")

rp_cat = (rpassive_D[rpassive_D["lipper_obj_cd"].isin(panel_cats) & rpassive_D["year_month"].between(ym_min, ym_max)]
          .merge(fac, on="year_month", how="left")
          .dropna(subset=FACTORS + ["rf", "r_passive"]))
rp_cat["rp_excess"] = rp_cat["r_passive"] - rp_cat["rf"]
fm_weight = active_panel.groupby("lipper_obj_cd").size()

rows = []
for cat, g in rp_cat.groupby("lipper_obj_cd"):
    if len(g) < MIN_MONTHS:
        continue
    res = sm.OLS(g["rp_excess"], sm.add_constant(g[FACTORS])).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    rows.append({"cat": cat, "alpha_ann_pct": res.params["const"] * 1200, "t_alpha": res.tvalues["const"],
                 "mkt_beta": res.params["mkt_rf"], "r2": res.rsquared, "n_months": len(g),
                 "active_fm": int(fm_weight.get(cat, 0))})
cat_alpha = pd.DataFrame(rows).sort_values("active_fm", ascending=False).reset_index(drop=True)

_pm = passive_D.dropna(subset=["mret"]).assign(ym=lambda d: d["caldt"].dt.to_period("M"))
med_funds = _pm.groupby(["ym", "lipper_obj_cd"])["crsp_fundno"].nunique().groupby("lipper_obj_cd").median()
DIVERSIFIED = {"G", "SG", "GI", "MC", "MR", "EI", "CA"}
cat_alpha["med_funds"] = cat_alpha["cat"].map(med_funds).round(1)
cat_alpha["trust"] = (cat_alpha["r2"] >= 0.90) & (cat_alpha["med_funds"] >= 15) & cat_alpha["cat"].isin(DIVERSIFIED)
print(cat_alpha.round(3).to_string(index=False))

wavg  = lambda d: np.average(d["alpha_ann_pct"], weights=d["active_fm"])
core  = cat_alpha[cat_alpha["trust"]]
print(f"\nFund-month-weighted passive alpha: all categories {wavg(cat_alpha):.3f}%/yr | "
      f"trusted core {wavg(core):.3f}%/yr ({', '.join(core['cat'])}; "
      f"range {core['alpha_ann_pct'].max():.2f} to {core['alpha_ann_pct'].min():.2f})")
assert_matches("Section 4.1: passive alpha, core mean / max / min (%/yr)",
               [wavg(core), core["alpha_ann_pct"].max(), core["alpha_ann_pct"].min()], [-1.1, -0.7, -1.8], 1)

**Section 5.4.1 — equal- vs value-weighted benchmark.** Cell-level (month × category) EW and VW returns on a common
sample (beginning-of-month TNA weights), so only the weighting differs.

In [ ]:
tmp = passive_D.sort_values(["crsp_fundno", "caldt"]).copy()
tmp["w"] = tmp.groupby("crsp_fundno")["mtna"].shift(1)
tmp = tmp[tmp["mret"].notna() & tmp["w"].notna() & (tmp["w"] > 0)].copy()
tmp["wr"] = tmp["w"] * tmp["mret"]

g = tmp.groupby(["caldt", "lipper_obj_cd"])
cell = pd.DataFrame({"r_ew": g["mret"].mean(), "r_vw": g["wr"].sum() / g["w"].sum()}).reset_index()
cell["diff"] = cell["r_vw"] - cell["r_ew"]

print(cell["diff"].describe(percentiles=[.05, .25, .5, .75, .95]).round(5).to_string())
print(f"\nMean VW−EW gap: {cell['diff'].mean()*100:.3f}%/month | share of cells with VW > EW: {(cell['diff'] > 0).mean():.1%}")
assert_matches("Section 5.4.1: mean VW−EW gap (%/month)", cell["diff"].mean() * 100, 0.044, 3)
assert (cell["diff"] > 0).mean() <= 0.5

**Section 5.4.2 — alternative cash instrument.** Replacing r<sub>f</sub> by a realised money-market return r<sub>cash</sub>
changes the corrected return by exactly Cash<sub>i,t</sub>·δ<sub>t</sub> with δ = r<sub>f</sub> − r<sub>cash</sub>, so the shift
in Δα̂ is bounded without re-estimation. r<sub>cash</sub> = equal-weighted net return of CRSP funds with Lipper code "MM".

In [ ]:
mm_ids = set(df2.loc[df2["lipper_obj_cd"] == "MM", "crsp_fundno"].unique())
r_cash = (monthly[monthly["crsp_fundno"].isin(mm_ids)].dropna(subset=["mret"])
          .groupby("caldt")["mret"].mean().rename("r_cash"))

rf_m  = merged_D.groupby("caldt")["rf"].first().rename("rf")
wedge = pd.concat([rf_m, r_cash], axis=1).dropna()
wedge["delta"] = wedge["rf"] - wedge["r_cash"]
d = wedge["delta"]

pert = merged_D[["caldt", "cash_weight"]].merge(wedge["delta"], left_on="caldt", right_index=True, how="inner")
p = (pert["cash_weight"] * pert["delta"]).dropna()

print(f"Money-market funds: {len(mm_ids)} | months: {len(d)}")
print(f"Wedge δ: mean {d.mean()*1200:+.3f}%/yr, positive in {(d > 0).mean():.0%} of months, "
      f"min/max {d.min()*1e4:+.1f}/{d.max()*1e4:+.1f} bps/month")
print(f"Induced shift in the correction (≈ shift in Δα̂): mean {p.mean()*1200:+.4f}%/yr, "
      f"99th pct |change| {p.abs().quantile(.99)*1200:.4f}%/yr, max |change| {p.abs().max()*1200:.1f}%/yr")

assert_matches("Section 5.4.2: MM funds, months", [len(mm_ids), len(d)], [852, 270], 0)
assert_matches("Section 5.4.2: mean wedge (%/yr), share positive", [d.mean() * 1200, (d > 0).mean()], [0.12, 0.75], 2)
assert_matches("Section 5.4.2: shift in Δα̂ (%/yr)", p.mean() * 1200, 0.001, 3)
assert_matches("Section 5.4.2: most negative wedge (bps), largest single change (%/yr)",
               [round(d.min() * 1e4), round(p.abs().max() * 1200)], [-230, 23], 0)